# Classification: Random Forest Classifier

## Justification of Preprocessing Strategy

### Scale Invariance

The Random Forest Classifier is an ensemble method composed of multiple Decision Trees. Since trees split data based on individual feature thresholds, they are scale-invariant. This means the model's logic is unaffected by whether the data is in its original range, standardized, or normalized. To preserve the clinical meaning of the features and maximize computational speed, we will use the Original Data.

### From Single Tree to Ensemble

While a single Decision Tree can be prone to overfitting, a Random Forest reduces variance by averaging the predictions of many trees. It introduces Feature Randomness (at each split, only a random subset of features is considered), which ensures that the trees are decorrelated and the model is more robust.

## Hyperparameter Strategy

In this experiment, we combine the structural knowledge gained from our Decision Tree "Champion" with the unique ensemble parameters of the Random Forest.

### Reusing Decision Tree Champion Parameters

We have identified that the following parameters provided the best results for a single tree, and we will use them as a baseline for our forest:

- `criterion='entropy'`: Entropy was found to be superior for measuring information gain in this clinical dataset.
- `max_depth=29`: This depth provides enough complexity to map diabetes patterns without memorizing noise.
- `min_samples_split=7` & `min_samples_leaf=3`: These act as pruning mechanisms, ensuring that splits are statistically significant and leaves represent a minimum number of patients.

### Random Forest Specific Parameters

To optimize the ensemble effect, we introduce these relevant parameters:

- `n_estimators`: The number of trees in the forest. We will test if increasing from 50 to 200 trees significantly improves our Recall.
- `max_features`: Controls how many features each tree evaluates at each split. This is the primary driver of tree diversity.
- `bootstrap`: Determines if trees are built using the entire dataset or random samples with replacement.

## Experiment Design

We defined a tournament of 3 optimization levels:

- Baseline: 100 trees using the exact parameters of the DTC Champion.
- GridSearchCV: A systematic search focusing on the forest's diversity (max_features) and the number of estimators.
- Optuna: Bayesian optimization to find the ultimate balance between individual tree depth and forest size, aiming for maximum Recall.

In [5]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_RandomForest")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Bagging/RanForClassifier/mlruns/8'), creation_time=1778148557516, experiment_id='8', last_update_time=1778148557516, lifecycle_stage='active', name='Classification_RandomForest', tags={}, workspace='default'>

In [6]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Transforming categorical variables via One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Features (X) and Target (y)
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_rf_metrics(y_true, y_pred, duration):
    """Log performance metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE RF 
# ---------------------------------------------------------
with mlflow.start_run(run_name="RF_Baseline"):
    # Initializing RF with parameters from our best Decision Tree
    rf_base = RandomForestClassifier(
        n_estimators=100,
        criterion='entropy',
        max_depth=29,
        min_samples_leaf=3,
        min_samples_split=7,
        random_state=42,
        n_jobs=-1
    )
    
    start_time = time.time()
    rf_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = rf_base.predict(X_test)
    
    mlflow.log_params(rf_base.get_params())
    mlflow.log_param("optimization", "none")
    log_rf_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV
# ---------------------------------------------------------
with mlflow.start_run(run_name="RF_GridSearch"):
    # Searching for optimal tree count and feature sampling strategy
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True, False]
    }
    
    grid = GridSearchCV(
        RandomForestClassifier(criterion='entropy', max_depth=29, min_samples_split=7, min_samples_leaf=3, random_state=42, n_jobs=-1),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred_grid = grid.best_estimator_.predict(X_test)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    log_rf_metrics(y_test, y_pred_grid, duration)
    best_rf_model = grid.best_estimator_

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False])
    }
    
    model = RandomForestClassifier(criterion='entropy', max_depth=29, min_samples_split=7, min_samples_leaf=3, **params, n_jobs=-1)
    # Using 3-fold cross-validation for optimization efficiency
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="RF_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15) 
    duration = time.time() - start_time
    
    # Train champion model with the discovered optimal parameters
    best_rf_opt = RandomForestClassifier(**study.best_params, n_jobs=-1)
    best_rf_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    log_rf_metrics(y_test, best_rf_opt.predict(X_test), duration)

[I 2026-05-07 11:27:17,937] A new study created in memory with name: no-name-38415ff4-a642-4132-a803-730040868219
[I 2026-05-07 11:27:22,072] Trial 0 finished with value: 0.8697862118986603 and parameters: {'n_estimators': 113, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 0.8697862118986603.
[I 2026-05-07 11:27:28,942] Trial 1 finished with value: 0.8692028551263621 and parameters: {'n_estimators': 161, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.8697862118986603.
[I 2026-05-07 11:27:32,433] Trial 2 finished with value: 0.8692653538241973 and parameters: {'n_estimators': 52, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 0.8697862118986603.
[I 2026-05-07 11:27:40,441] Trial 3 finished with value: 0.869182020490864 and parameters: {'n_estimators': 138, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 0.8697862118986603.
[I 2026-05-07 11:27:51,430] Trial 4 finished with value: 0.869

## Runs Summary

| Run | Optimization | n_estimators | max_features | bootstrap | Accuracy | F1 | Recall | Fit Time |
|---|---|---:|---|---|---|---:|---:|---:|
| RF_Baseline | none | 100 | sqrt | True | 0.91965 | 0.9282749386 | 0.8665833333 | 2.33s |
| RF_GridSearch | GridSearchCV | 50 | log2 | True | 0.91865 | 0.9274599848 | 0.86675 | 67.01s |
| RF_Optuna | optuna | 107 | log2 | True | 0.9188 | 0.9276227828 | 0.86725 | 95.46s |

### Additional logged parameters
- **Fixed across all runs**: `criterion='entropy'`, `max_depth=29`, `min_samples_split=7`, `min_samples_leaf=3`
- `random_state = 42` in all runs
- `n_jobs = -1` in all runs
- **Optimized parameters** (GridSearch & Optuna): `n_estimators`, `max_features`, `bootstrap`

## Best Run Justification for Streamlit

The best run to use in Streamlit is **RF_Baseline**. **RF_Optuna** shows the highest recall (0.86725), followed by **RF_GridSearch** (0.86675) and the baseline (0.8665833333). However, this recall improvement is small and comes with a very large increase in training time: 2.33s for the baseline, 67.01s for GridSearch, and 95.46s for Optuna.

In a clinical setting like this, maximizing recall alone is not sufficient. It is necessary to balance the ability to identify patients with diabetes with overall model stability and the efficiency required for Streamlit deployment. On that balance, the baseline is the best compromise because it provides:
- the highest **accuracy** (0.91965);
- the highest **F1-score** (0.9282749386), indicating the best balance between precision and recall;
- a **recall** very close to the best obtained, without sacrificing overall performance;
- the shortest **training time** by a large margin, which is decisive for integration and maintenance in production;
- the same fixed parameters used across runs, keeping comparisons fair.

Therefore, **RF_Baseline** offers the best trade-off between performance, interpretability, and computational efficiency for predicting whether a patient has diabetes in Streamlit.